In [78]:
import json

In [79]:
with open("data/인터뷰/ckmk_d_bm_f_n_169087.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

In [80]:
from pprint import pprint

In [81]:
pprint(data)

{'dataSet': {'answer': {'emotion': [],
                        'intent': [{'category': 'attitude',
                                    'expression': '',
                                    'text': ''}],
                        'raw': {'text': '저는 저 혼자 시간을 보내는 것을 우선적으로 하고요. 예를 들어서 '
                                        '혼자 산책을 한다든지 아니면 혼자 카페를 간다든지 아니면 혼자 코인 '
                                        '노래방을 간다든지 해서 스트레스를 해소하는 편입니다. 물론 남들과 '
                                        '만나면서 시간을 보내는 것도 굉장히 좋아하지만요. 아무래도 내 '
                                        '혼자만의 시간이 있어야 어느 정도 생각 정리도 되고 그러면서 내 '
                                        '안에 있던 응어리도 절로 풀러지는 그런 것을 경험하게 되었습니다. '
                                        '그리고 코인 노래방이나 이런 곳에 가면은 아무래도 제가 노래 부르는 '
                                        '것도 좋아하고 그리고 노래를 부르면서 그런 소리가 웅웅거리고 또 '
                                        '소리를 지른다는 거에 대해서 해소감도 있는 것 같고요. 그래서 저는 '
                                        '일단 혼자만의 시간을 보내는데 뭐 혼자서 노래방을 가거나 아니면 '

In [82]:
text =  data['dataSet']['answer']['raw']['text']
summary = data['dataSet']['answer']['summary']['text']

In [83]:
text

'저는 저 혼자 시간을 보내는 것을 우선적으로 하고요. 예를 들어서 혼자 산책을 한다든지 아니면 혼자 카페를 간다든지 아니면 혼자 코인 노래방을 간다든지 해서 스트레스를 해소하는 편입니다. 물론 남들과 만나면서 시간을 보내는 것도 굉장히 좋아하지만요. 아무래도 내 혼자만의 시간이 있어야 어느 정도 생각 정리도 되고 그러면서 내 안에 있던 응어리도 절로 풀러지는 그런 것을 경험하게 되었습니다. 그리고 코인 노래방이나 이런 곳에 가면은 아무래도 제가 노래 부르는 것도 좋아하고 그리고 노래를 부르면서 그런 소리가 웅웅거리고 또 소리를 지른다는 거에 대해서 해소감도 있는 것 같고요. 그래서 저는 일단 혼자만의 시간을 보내는데 뭐 혼자서 노래방을 가거나 아니면 혼자서 카페를 가거나 그런 식으로 스트레스를 해소한다 라고 말씀을 드릴 수 있을 것 같습니다.'

In [84]:
from glob import glob

In [85]:
file_list = glob('data/인터뷰/*.json')

In [86]:
texts , summarys = [], []

for idx, file in enumerate(file_list):
    try:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        text =  data['dataSet']['answer']['raw']['text']
        summary = data['dataSet']['answer']['summary']['text']
        texts.append(text)
        summarys.append(summary)
    except:
        continue
    if idx == 1000:
        break

In [87]:
summarys[0]

'저는 혼자 시간을 보내는 것을 우선적으로 하며, 혼자 산책을 하거나 혼자 카페를 간다든지 또는 혼자 노래방을 간다든지 해서 스트레스를 해소하는 편입니다. 혼자의 시간이 있으면 생각 정리도 되고 제 안에 있던 응어리도 저절로 풀러지는 것을 경험하게 되었습니다.'

In [88]:

import numpy as np
from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

In [89]:
# ------------------------------------------------------------
# 0) 모델 이름 지정 (KoBART)
# ------------------------------------------------------------
MODEL_NAME = "gogamza/kobart-base-v2"   # 한국어 BART 모델

In [90]:
train_docs = texts[ : 100]
train_sums = summarys[ : 100]
valid_docs = texts[-10 : ]
valid_sums = summarys[-10 : ]

In [91]:
# HuggingFace datasets 형식으로 변환
raw_ds = DatasetDict({
    "train": Dataset.from_dict({"document": train_docs, "summary": train_sums}),
    "validation": Dataset.from_dict({"document": valid_docs, "summary": valid_sums}),
})
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 10
    })
})

In [92]:

# ------------------------------------------------------------
# 2) 토크나이저 / 모델 로드
#    KoBART는 SentencePiece 기반이므로 반드시 sentencepiece 설치 필요
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# 입력/출력 문장 최대 길이
max_input_len = 512
max_target_len = 128


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


In [93]:

# ------------------------------------------------------------
# 3) 데이터 전처리 함수
#    Trainer는 tokenized 데이터셋을 요구 → map()으로 변환
# ------------------------------------------------------------
def preprocess_fn(batch):
    # ------------------------------
    # (1) 입력 문서 인코딩
    # ------------------------------
    inputs = tokenizer(
        batch["document"],
        max_length=max_input_len,
        padding="max_length",    # 고정 길이 패딩
        truncation=True,         # 길면 자르기
    )

    # ------------------------------
    # (2) 요약(정답) 인코딩
    #     → target tokenization
    #     → pad token은 -100으로 바꿔 loss 계산에서 제외
    # ------------------------------
    # 디코더용 토큰나이저를 사용
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["summary"],
            max_length=max_target_len,
            padding="max_length",
            truncation=True,
        )

    # pad 토큰을 -100으로 변경 (CrossEntropyLoss ignore_index)
    labels_ids = np.array(labels["input_ids"])
    labels_ids[labels_ids == tokenizer.pad_token_id] = -100
    inputs["labels"] = labels_ids.tolist()

    return inputs

# raw_ds에 전처리 적용
tokenized_ds = raw_ds.map(
    preprocess_fn,
    batched=True,
    remove_columns=["document", "summary"]   # 원본 텍스트는 삭제
)

# ------------------------------------------------------------
# 4) DataCollator
#    padding 동적 처리 + 모델 입력 형태 자동 구성
# ------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 10/10 [00:00<00:00, 2916.36 examples/s]


In [94]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10
    })
})

In [ ]:

# ------------------------------------------------------------
# 5) ROUGE Metric 설정
#    (요약 모델에서는 텍스트 생성 지표 필요)
# ------------------------------------------------------------
rouge = evaluate.load("rouge")

def postprocess_text(texts):
    # Rouge 계산을 위해 텍스트 양쪽 공백 정리
    return [t.strip() for t in texts]


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # -100(ignored index)을 다시 pad token으로 복원해야 디코딩 가능
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # 텍스트 디코딩
    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_str = postprocess_text(pred_str)
    label_str = postprocess_text(label_str)

    # ROUGE 계산
    result = rouge.compute(
        predictions=pred_str,
        references=label_str,
        use_stemmer=True
    )
    # use_stemmer : ROUGE 점수를 계산할 때 단어의 “원형(어간)” 기준으로 비교해라는 의미

    # % 단위로 보기 쉽게 소수점 정리
    result = {k: round(v * 100, 2) for k, v in result.items()}
    return result

# ------------------------------------------------------------
# 6) 학습 설정 (Training Arguments)
# ------------------------------------------------------------
args = Seq2SeqTrainingArguments(
    output_dir="./kobart-sum",          # 모델 저장 경로
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=10,

    eval_strategy="epoch",        # epoch마다 평가
    save_strategy="epoch",              # epoch마다 모델 저장

    # ✅ generate() 관련 옵션들 (여기!)
    predict_with_generate=True,
    generation_max_length=120,
    # 요약을 생성할 때 몇 개의 “경로(문장 후보)”를 동시에 탐색할지를 결정하는 옵션
    generation_num_beams=5,


    load_best_model_at_end=True,        # 가장 좋은 모델 자동 로드
    metric_for_best_model="rougeL",     # ROUGE-L 기준
    greater_is_better=True,             # 값이 클수록 좋음

    report_to="none",                   # wandb/logging 비활성
)

# ------------------------------------------------------------
# 7) Trainer 객체 생성
# ------------------------------------------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ------------------------------------------------------------
# 8) 학습 시작
# ------------------------------------------------------------
trainer.train()


C:\Users\ekfla\AppData\Local\Temp\ipykernel_34424\515920190.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.451300,1.601245,0.000000,0.000000,0.000000,0.000000


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=50, training_loss=0.2783675193786621, metrics={'train_runtime': 84.1036, 'train_samples_per_second': 1.189, 'train_steps_per_second': 0.595, 'total_flos': 30486822912000.0, 'train_loss': 0.2783675193786621, 'epoch': 1.0})

In [106]:

# ------------------------------------------------------------
# 9) 검증 데이터 평가
# ------------------------------------------------------------
metrics = trainer.evaluate()
print("\n[Validation Metrics]", metrics)

# ------------------------------------------------------------
# 10) 파인튜닝된 모델로 직접 요약 생성 테스트
# ------------------------------------------------------------
test_text = texts[130]
test_sum = summarys[130]
inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=max_input_len
)

inputs.pop("token_type_ids", None)  # 완전 제거
# 입력 인코딩 후 모델에 전달
gen_ids = model.generate(
    **inputs.to(model.device),
    max_new_tokens=80,      # 출력 토큰 길이
    min_new_tokens=10,      # 너무 일찍 끝내지 않기
    num_beams=4,            # 4개의 후보 문장을 병렬적으로 추적하여 가장 가능성이 높은 출력 선택.
    do_sample=False,        #샘플링 사용 여부. False → 결정적(greedy/beam search) 생성 True → 확률에 따라 랜덤 생성(top-k, top-p 적용 가능)
    length_penalty=1.6,     # 생성 길이에 대한 가중치.
    no_repeat_ngram_size=5, # 반복 방지(필요시) 3-gram(연속 3토큰) 반복을 금지.
    early_stopping=True,    # 모든 빔이 EOS에 도달하면 조기 종료
    repetition_penalty=1.5, # 토큰 반복 패턴이 나타나면 확률을 1.2배 패널티 적용하여 반복을 억제
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

print("\n[원본 문장]")
print(test_text)
print("\n[실제 요약]")
print(test_sum)
print("\n[생성 요약]")
print(tokenizer.decode(gen_ids[0], skip_special_tokens=True))


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



[Validation Metrics] {'eval_loss': 1.6012451648712158, 'eval_rouge1': 0.0, 'eval_rouge2': 0.0, 'eval_rougeL': 0.0, 'eval_rougeLsum': 0.0, 'eval_runtime': 25.3738, 'eval_samples_per_second': 0.394, 'eval_steps_per_second': 0.197, 'epoch': 1.0}

[원본 문장]
일을 하다보면 업무량이 과도하게 많아질 수 있다고 하셨는데요. 어 저는 직급이나 어떤 역할이 많아지게 되면 업무량이 많아지게 되는 것은 어쩔 수 없는 수순이라고 생각합니다. 그렇기 때문에 어느 정도로 업무량이 늘어난다면 그것은 제가 감내해야 할 부분이라고 생각을 합니다. 하지만 제가 제 체력과 그리고 제 삶이 무너지지 않는 선에서 그것을 해결할 수 있는 수준이었으면 좋겠구요. 만약에 그것이 너무 저 저에게 과도하게 부담이 되고 그것 때문에 제가 어 몸에 병이 오거나 하는 수준에 부담으로 다가오게 된다면 그것은 어느정도 조정을 해야 할 것이고 어 그런 부분이 이 회사 내에서 조정이 되지 않는다면 절대적인 인력의 부족이기 때문에 인력을 확충해 달라고 말씀을 드려 볼 것 같습니다. 어 개인이 행복해야 회사도 결국에는 발전이 있는 것이라고 생각하기 때문에 충분히 받아들여 주실 거라고 생각합니다.

[실제 요약]
업무량이 과도한 것은 어쩔 수 없는 수순이지만, 체력과 삶이 무너지지 않는 선에서 그것을 해결할 수 있었으면 좋겠습니다. 만약 그것이 회사 내에서 조정이 되지 않는다면 인력을 확충해 달라고 말씀을 드려 볼 것 같습니다. 개인이 행복해야 회사도 발전이 있는 것이라고 생각하기 때문에 충분히 받아들여 주실 것이라 생각합니다.

[생성 요약]
업무량이 많아지게 되면 업무량이 많아지게 되는 것은 어쩔 수 없는 수순이라고 생각합니다. 하지만 제 체력과 삶이 무너지지 않는 선에서 그것을 해결할 수 있는 수준이었으면 합니다. 그것